# Hallmark Python Demo

Hallmark helps you keep track of data files in scientific workflows.
It stores a file index inside a small git repository (`.hm`) that lives
next to your data, so you always know which files belong to which run.

This notebook walks through the full workflow:
1. Create a hallmark repository
2. Add files and commit
3. Discover and filter files with `ParaFrame`
4. Handle custom filename conventions with encoding

In [1]:
# auto-reload modules when they change
%load_ext autoreload
%autoreload 2

from hallmark import Repo, ParaFrame

In [2]:
# remove any repositories left over from a previous run
! rm -rf repo*

## 1. Create a Repository

`Repo.init(path)` creates a new hallmark repository at `path`.
It works like `git init`: the directory is created with a `.hm` folder
inside, which is itself a git repository used to store the file manifest.
The shared `objects/` store is created lazily on the first commit that stores
file bytes.


In [3]:
repo1 = Repo.init("repo1")

In [4]:
# repo1/ holds your data files; repo1/.hm/ holds the hallmark index
! ls -a repo1 repo1/.hm

repo1:
.   ..  .hm

repo1/.hm:
.          ..         .git       config.yml data.tsv   meta.yml   README.md


In [5]:
# the .hm directory is a git repo — you can run git commands inside it
! pushd repo1/.hm && git status && popd

On branch master
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   config.yml
	new file:   data.tsv
	new file:   meta.yml



The `.hm` repository already has an initial README commit. The hallmark state
files (`config.yml`, `data.tsv`, `meta.yml`) are then staged in the working tree
so the next `Repo.commit(...)` captures the first manifest snapshot.


### Bare repository

A bare repository has no data directory — it is just the index.
Name the path with a `.hm` suffix to create one.

In [6]:
repo2 = Repo.init("repo2.hm")

In [7]:
# repo2.hm/ is the index itself — there is no separate data folder
! ls -a repo2.hm

.          ..         .git       config.yml data.tsv   meta.yml   README.md


In [8]:
! pushd repo2.hm && git status && popd

On branch master
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   config.yml
	new file:   data.tsv
	new file:   meta.yml



## 2. Add and Commit Files

`repo.add(fmt)` scans `repo1/` for files whose names match the format
string, computes a checksum for each one, and stages them in the manifest.
It returns a table showing the matched files and their parsed parameters.

`repo.commit(msg)` saves the current state of the manifest as a git commit
inside `.hm`, and stores the corresponding file bytes in the shared `objects/`
store.


In [9]:
%%bash
# create sample data files: spin values 0, 0.75, 0.94 at inclinations 0–180
pushd repo1
for f in a{0,0.75,0.94}_i{0,30,60,90,120,150,180}.h5; do echo "$f" > "$f"; done
popd
ls repo1

~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallma

rk/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research

/github/hallmark/demo


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallma

rk/demo


a0_i0.h5
a0_i120.h5
a0_i150.h5
a0_i180.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0

.75_i150.h5
a0.75_i180.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5
a0.94_i0.h5
a0.94_i120.h5
a0.94_i15

0.h5
a0.94_i180.h5
a0.94_i30.h5
a0.94_i60.h5
a0.94_i90.h5


In [10]:
# scan repo1/ for files matching the pattern and record them in the index
pf = repo1.add("a{a}_i{i}.h5")
pf

,path,a,i,sha1
0,a0.75_i0.h5,0.75,0.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
1,a0.75_i120.h5,0.75,120.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
2,a0.75_i150.h5,0.75,150.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
3,a0.75_i180.h5,0.75,180.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
4,a0.75_i30.h5,0.75,30.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
5,a0.75_i60.h5,0.75,60.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
6,a0.75_i90.h5,0.75,90.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
7,a0.94_i0.h5,0.94,0.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
8,a0.94_i120.h5,0.94,120.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
9,a0.94_i150.h5,0.94,150.0,da39a3ee5e6b4b0d3255bfef95601890afd80709


In [11]:
# save this snapshot — returns True if a commit was made
repo1.commit("add spin and inclination files")

True

## 3. Discover and Filter Files with ParaFrame

`repo.add()` uses `ParaFrame` under the hood to find files.
You can also call `ParaFrame.parse()` directly — useful when you want to
look up files without adding them to the index.

`ParaFrame` is a table (a pandas DataFrame) where each row is a file and
each column is a parameter read from the file name.

In [12]:
# discover files in repo1/ directly — no index update
pf = ParaFrame.parse("a{a}_i{i}.h5", base_path="repo1")
pf

,path,a,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i150.h5,0.75,150.0
3,a0.75_i180.h5,0.75,180.0
4,a0.75_i30.h5,0.75,30.0
5,a0.75_i60.h5,0.75,60.0
6,a0.75_i90.h5,0.75,90.0
7,a0.94_i0.h5,0.94,0.0
8,a0.94_i120.h5,0.94,120.0
9,a0.94_i150.h5,0.94,150.0


### Debug mode

`debug=True` prints the glob pattern used to search for files.
This helps you check the pattern when no files are found.

In [13]:
ParaFrame.parse("a{a}_i{i}.h5", base_path="repo1", debug=True)

0 repo1/a{a}_i{i}.h5 () {}
1 repo1/a{a:s}_i{i}.h5 () {'a': '*'}
2 repo1/a{a:s}_i{i:s}.h5 () {'a': '*', 'i': '*'}
Pattern: "repo1/a*_i*.h5"
21 matches, e.g., "repo1/a0.75_i0.h5"


,path,a,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i150.h5,0.75,150.0
3,a0.75_i180.h5,0.75,180.0
4,a0.75_i30.h5,0.75,30.0
5,a0.75_i60.h5,0.75,60.0
6,a0.75_i90.h5,0.75,90.0
7,a0.94_i0.h5,0.94,0.0
8,a0.94_i120.h5,0.94,120.0
9,a0.94_i150.h5,0.94,150.0


### Filter by parameter value

Call a `ParaFrame` like a function to filter rows:
- **scalar** — keep rows where the column equals that value
- **list** — keep rows where the column matches any value in the list
- **chaining** — each call narrows the result further (AND logic)

In [14]:
# keep only rows where a == 0
pf(a=0)

,path,a,i
14,a0_i0.h5,0.0,0.0
15,a0_i120.h5,0.0,120.0
16,a0_i150.h5,0.0,150.0
17,a0_i180.h5,0.0,180.0
18,a0_i30.h5,0.0,30.0
19,a0_i60.h5,0.0,60.0
20,a0_i90.h5,0.0,90.0


In [15]:
# keep rows where a == 0 or a == 0.75
pf(a=[0, 0.75])

,path,a,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i150.h5,0.75,150.0
3,a0.75_i180.h5,0.75,180.0
4,a0.75_i30.h5,0.75,30.0
5,a0.75_i60.h5,0.75,60.0
6,a0.75_i90.h5,0.75,90.0
14,a0_i0.h5,0.00,0.0
15,a0_i120.h5,0.00,120.0
16,a0_i150.h5,0.00,150.0


In [16]:
# keep rows where a == 0.94 AND i == 0  (chain two calls)
pf(a=0.94)(i=0)

,path,a,i
7,a0.94_i0.h5,0.94,0.0


In [17]:
# use standard pandas boolean indexing for range or custom conditions
pf[pf.i <= 60]

,path,a,i
0,a0.75_i0.h5,0.75,0.0
4,a0.75_i30.h5,0.75,30.0
5,a0.75_i60.h5,0.75,60.0
7,a0.94_i0.h5,0.94,0.0
11,a0.94_i30.h5,0.94,30.0
12,a0.94_i60.h5,0.94,60.0
14,a0_i0.h5,0.00,0.0
18,a0_i30.h5,0.00,30.0
19,a0_i60.h5,0.00,60.0


### Use filtered paths in a loop

Note: passing multiple keyword arguments in a single `pf(...)` call gives
OR logic across all of them. Chain calls to get AND logic.

In [18]:
# selects rows where a == 0 OR i == 0  (not both at once)
for path in pf(a=0, i=0).path:
    print(f'Processing "{path}"...')

Processing "a0.75_i0.h5"...
Processing "a0.94_i0.h5"...
Processing "a0_i0.h5"...
Processing "a0_i120.h5"...
Processing "a0_i150.h5"...
Processing "a0_i180.h5"...
Processing "a0_i30.h5"...
Processing "a0_i60.h5"...
Processing "a0_i90.h5"...


## 4. Custom Filename Encoding

Some datasets use a special convention to write certain values in file names.
For example, a negative spin value like `-0.94` might be stored as `m0.94`
in the filename, where the `m` prefix means "minus".

Hallmark handles this with an encoding config: you write a regular expression
that describes the pattern, and `hallmark` applies it before reading the
parameter values. You store this config in `.hm/config.yml`.

### Create files and add the encoding config

In [19]:
%%bash
# create files with negative spin values using the m-prefix convention
pushd repo1
for f in am0.75_i0.h5 am0.75_i30.h5 am0.75_i60.h5          am0.94_i0.h5 am0.94_i30.h5 am0.94_i60.h5; do echo "$f" > "$f"; done
popd
ls repo1/am*.h5

~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallma

rk/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research

/github/hallmark/demo


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallma

rk/demo


repo1/am0.75_i0.h5
repo1/am0.75_i30.h5
repo1/am0.75_i60.h5
repo1/am0.94_i0.h5
repo1/am0.94_i30.h5
re

po1/am0.94_i60.h5


In [20]:
%%bash
# write the encoding config to .hm/config.yml
# the regex m([0-9]+(\.[0-9]+)?) matches the m-prefix and captures the number
cat > repo1/.hm/config.yml <<'EOF'
encodings:
  - fmt: "a{aspin}_i{i}.h5"
    encoding:
      aspin: "m([0-9]+(\\.[0-9]+)?)"
EOF

In [21]:
# reload the config so the repo object knows about the new encoding
repo1.state.config = repo1.dothm.load_yml("config")

In [22]:
# add all files — m0.94 in the filename is decoded to -0.94 in the table
pf_spin = repo1.add("a{aspin}_i{i}.h5", encoding=True)
pf_spin

,path,aspin,i,sha1
0,a0.75_i0.h5,0.75,0.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
1,a0.75_i120.h5,0.75,120.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
2,a0.75_i150.h5,0.75,150.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
3,a0.75_i180.h5,0.75,180.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
4,a0.75_i30.h5,0.75,30.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
5,a0.75_i60.h5,0.75,60.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
6,a0.75_i90.h5,0.75,90.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
7,a0.94_i0.h5,0.94,0.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
8,a0.94_i120.h5,0.94,120.0,da39a3ee5e6b4b0d3255bfef95601890afd80709
9,a0.94_i150.h5,0.94,150.0,da39a3ee5e6b4b0d3255bfef95601890afd80709


In [23]:
repo1.commit("add negative spin files with encoding")

True

### Discover encoded files with ParaFrame

You can also use `ParaFrame.parse()` directly with encoding.
Pass the encoding config from `repo1.state.config` so it knows the regex.

In [24]:
encodings = repo1.state.config.get("encodings", [])

pf_spin = ParaFrame.parse(
    "a{aspin}_i{i}.h5",
    encodings=encodings,
    base_path="repo1",
    encoding=True,
)
pf_spin

,path,aspin,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i150.h5,0.75,150.0
3,a0.75_i180.h5,0.75,180.0
4,a0.75_i30.h5,0.75,30.0
5,a0.75_i60.h5,0.75,60.0
6,a0.75_i90.h5,0.75,90.0
7,a0.94_i0.h5,0.94,0.0
8,a0.94_i120.h5,0.94,120.0
9,a0.94_i150.h5,0.94,150.0


In [25]:
# filter using the decoded float values — not the raw m-prefix strings
pf_spin(aspin=[-0.75, -0.94])

,path,aspin,i
21,am0.75_i0.h5,-0.75,0.0
22,am0.75_i30.h5,-0.75,30.0
23,am0.75_i60.h5,-0.75,60.0
24,am0.94_i0.h5,-0.94,0.0
25,am0.94_i30.h5,-0.94,30.0
26,am0.94_i60.h5,-0.94,60.0
